# IBM Quantum Workshop

1. Set your options in the **SETUP** cell below, then run it.
2. Run the exercises in order. Each one shows you its circuit, then runs it.

Every cell is yours to edit — change a number, re-run, see what happens.

In [ ]:
# ============ SETUP — edit these, then run this cell ============

USE_SIMULATOR = True   # True = local simulator, False = your real IBM Quantum instance

# Only needed if USE_SIMULATOR = False:
API_KEY = ""           # 44-character API key from your IBM Quantum dashboard
CRN = ""               # starts with crn:v1:bluemix... (Instances page)


# --- nothing to change below this line ---
from workshop import setup

setup(USE_SIMULATOR, API_KEY, CRN)

For the simulator there is nothing to fill in — just run the cell.

For real hardware, set `USE_SIMULATOR = False`, paste your API key and CRN between
the quotes, and run the cell. If something is wrong you get a specific message right
there, instead of a failure three cells later.

---
# Exercise 1 — Tilting a qubit

A classical bit is 0 or 1. A qubit can sit *between* them, and you only find out
which one it is when you measure it.

The `ry` gate tilts the qubit. How far you tilt it sets how likely you are to
measure a `1`:

$$P(1) = \sin^2(\theta/2)$$

Tilt it a quarter turn and you get a fair coin. Tilt it further and `1` becomes
more likely. **Change `P_ONE` below and re-run the cell** to feel the difference.

In [ ]:
# ============ EXERCISE 1 — pick how biased your qubit is ============

P_ONE = 0.50    # chance of measuring 1.  Try: 0.10, 0.25, 0.50, 0.75, 0.90

# ---------------------------------------------------------------
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from IPython.display import display

from workshop import run_circuit

# P(1) = sin^2(theta/2), so this is the tilt that gives the probability above.
theta = 2 * np.arcsin(np.sqrt(P_ONE))

qubit = QuantumCircuit(QuantumRegister(1, 'q'), ClassicalRegister(1, 'meas'))
qubit.ry(theta, 0)      # tilt it
qubit.measure(0, 0)     # look at it

print(f'Tilting by {theta:.3f} radians -> should measure 1 about {P_ONE:.0%} of the time')
display(qubit.draw('mpl'))

# Measure the SAME circuit five separate times.
print('\nFive measurements of the same qubit:')
for i in range(5):
    counts = run_circuit(qubit, shots=1)
    outcome = list(counts)[0]
    print(f'   measurement {i + 1}:  {outcome}', flush=True)

Five measurements is a tiny sample — with `P_ONE = 0.5` you might still get five
`1`s in a row, the same way five coin flips can all come up heads.

Run it again for a different five. Then run the cell below, which measures the
*same* circuit 2000 times, to see the bias you asked for actually show up.

In [ ]:
# ============ EXERCISE 1b — the same qubit, measured 2000 times ============
from qiskit.visualization import plot_histogram

counts = run_circuit(qubit, shots=2000)

measured = counts.get('1', 0) / sum(counts.values())
print(f'asked for P(1) = {P_ONE:.0%}')
print(f'actually measured  = {measured:.1%}   from {counts}')

display(plot_histogram(counts))

---
# Exercise 2 — Entanglement

Two qubits this time. A Hadamard puts the first into superposition, then a CNOT
ties the second to it.

Now the strange part: measure them and you get `00` or `11` — but never `01` or
`10`. Each qubit on its own is still a fair coin, yet they always agree.

In [ ]:
# ============ EXERCISE 2 — a Bell pair ============
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from IPython.display import display

from workshop import run_circuit

bell = QuantumCircuit(QuantumRegister(2, 'q'), ClassicalRegister(2, 'meas'))
bell.h(0)               # first qubit into superposition
bell.cx(0, 1)           # tie the second one to it
bell.measure([0, 1], [0, 1])

display(bell.draw('mpl'))

counts = run_circuit(bell, shots=1024)
print('Counts:', counts)
print('Never saw 01 or 10:', not ({'01', '10'} & set(counts)))

display(plot_histogram(counts))

---
# Exercise 3 — Shor's algorithm

Factoring 15. This is the algorithm that breaks RSA — at a scale far beyond
today's hardware.

The quantum part finds the *period* of $a^x \bmod N$. Once you know the period,
the factors drop out of an ordinary gcd.

In [ ]:
# ============ EXERCISE 3 — Shor's algorithm ============

N = 15          # the number to factor
a = 7           # try 2, 4, 7, 8, 11 or 13
N_COUNT = 3     # counting qubits

# ---------------------------------------------------------------
from fractions import Fraction
from math import gcd

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from IPython.display import display

from workshop import run_circuit


def c_amod15(a, power):
    """Controlled multiplication by a^power mod 15.

    Hard-coded for 15: each valid `a` just permutes the four work qubits,
    so the modular arithmetic is a pattern of swaps.
    """
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        if a in [7, 8]:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        if a in [4, 11]:
            U.swap(1, 3); U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    gate = U.to_gate()
    gate.name = f'{a}^{power} mod 15'
    return gate.control()


def iqft(n):
    """Inverse quantum Fourier transform: turns hidden phases into a readable number."""
    qc = QuantumCircuit(n)
    for q in range(n // 2):
        qc.swap(q, n - 1 - q)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / 2 ** (j - m), m, j)
        qc.h(j)
    gate = qc.to_gate()
    gate.name = 'IQFT'
    return gate


# Build the period-finding circuit.
shor = QuantumCircuit(
    QuantumRegister(N_COUNT, 'count'),
    QuantumRegister(4, 'work'),
    ClassicalRegister(N_COUNT, 'meas'),
)
for q in range(N_COUNT):
    shor.h(q)
shor.x(N_COUNT)                     # work register starts at |1>
for q in range(N_COUNT):
    shor.append(c_amod15(a, 2 ** q), [q] + list(range(N_COUNT, N_COUNT + 4)))
shor.append(iqft(N_COUNT), range(N_COUNT))
shor.measure(range(N_COUNT), range(N_COUNT))

display(shor.draw('mpl', fold=-1))

counts = run_circuit(shor, shots=1024)
print('Measured phases:', counts)

# Classical part: phase -> period -> factors.
factors = set()
for bits in sorted(counts, key=counts.get, reverse=True):
    phase = int(bits, 2) / 2 ** N_COUNT
    r = Fraction(phase).limit_denominator(N).denominator
    if r % 2 == 0:
        for f in (gcd(a ** (r // 2) - 1, N), gcd(a ** (r // 2) + 1, N)):
            if f not in (1, N):
                factors.add(f)

if factors:
    print(f'\nFactors of {N}: {sorted(factors)}')
else:
    print('\nNo non-trivial factor this time (noise/luck) - run it again!')